# 🧠 GIADA — CC-05–CC-08 · Cella unificata e memoria compressa
Nove bracci × tre seed, target NEURON nativi, stesso supporto e stream. Stato esplicito, scala latente, tre bias di uscita e inizializzazione. Core ionico senza sinapsi: nessuna conferma globale o claim di accelerazione.
**Nessun dataset da aggiungere. GPU CUDA e Internet richiesti.** Codice, notebook e preregistrazione devono appartenere alla stessa revisione. Native e CUDA in subprocessi distinti; output limitato.

In [ ]:
import importlib.metadata, json, os, shutil, subprocess, sys
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
WORK=Path('/kaggle/working/giada_cc_n2');WORK.mkdir(parents=True,exist_ok=True)
REPO,TEACHER=WORK/'giada',WORK/'neuron_as_deep_net'
BRANCH='codex/surrogate-validity-audit'
if not REPO.exists():subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
REF=os.environ.get('GIADA_CC_N2_REVISION',BRANCH)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REF],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if not TEACHER.exists():subprocess.run(['git','clone','--depth','1','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
TEACHER_REV='074c4666300a8ad246601dab179a97a6942f0f29'
subprocess.run(['git','-C',str(TEACHER),'fetch','--depth','1','origin',TEACHER_REV],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach',TEACHER_REV],check=True)
try:installed=importlib.metadata.version('NEURON').split('+')[0]
except importlib.metadata.PackageNotFoundError:installed=None
if installed!='8.2.7':subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
assert shutil.which('nrnivmodl'),'Compilatore NEURON non disponibile'
import torch
assert torch.cuda.is_available(),'CUDA GPU richiesta'
SPEC=json.loads((REPO/'experiments/cc_n2_unified_cell_preregistration_v1.json').read_text())
assert SPEC['roadmap_ids']==['CC-05','CC-06','CC-07','CC-08']
print({'revision':REVISION,'gpu':torch.cuda.get_device_name(),'tasks':SPEC['roadmap_ids'],'arms':len(SPEC['arms']),'seeds':SPEC['seeds'],'inputs_required':0})


## 🧪 Matrice e freeze
Fit/development/diagnostica hanno episodi e preparazioni indipendenti. Checkpoint 0/150/500/1500, scelta su development; diagnostica soltanto dopo freeze. L'encoder riceve gate una volta; warmup solo storia causale V/Ca/input nei primi 10 ms. Rollout successivi autonomi di 8 e 30 ms. Il controllo esplicito ha loss gate ausiliaria: non è un confronto loss-matched.


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_cc_n2_unified_cell')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}; avvia una sessione pulita.'
LOG=Path('/kaggle/working/cc_n2_process.log')
command=[sys.executable,'-u',str(REPO/'scripts/run_cc_n2_unified_cell.py'),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
with subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process,LOG.open('w') as log:
    for line in process.stdout:
        log.write(line);log.flush()
        if line.startswith('[GIADA CC-N2]'):print(line.rstrip(),flush=True)
    RETURN_CODE=process.wait()
OUTPUT.mkdir(parents=True,exist_ok=True);shutil.copy2(LOG,OUTPUT/'process.log')
report_path=next((OUTPUT/n for n in ('final_report.json','failure_report.json') if (OUTPUT/n).is_file()),None)
REPORT=json.loads(report_path.read_text()) if report_path else {'valid':False,'diagnosis':'PROCESS_TERMINATED_WITHOUT_REPORT','exit':RETURN_CODE}
if report_path is None:(OUTPUT/'failure_report.json').write_text(json.dumps(REPORT,indent=2))
print({k:REPORT.get(k) for k in ('valid','diagnosis','error','member_count','new_candidate_promoted')})
if RETURN_CODE:print('\n'.join(LOG.read_text(errors='replace').splitlines()[-18:]))
ARCHIVE=shutil.make_archive('/kaggle/working/giada_cc_n2_unified_cell','zip',root_dir=OUTPUT)
print({'archive':ARCHIVE,'exit':RETURN_CODE})


In [ ]:
if (OUTPUT/'evaluation.json').is_file():
    import pandas as pd
    from IPython.display import display
    frame=pd.DataFrame(json.loads((OUTPUT/'evaluation.json').read_text()))
    display(frame.groupby(['role','horizon_ms','arm'])[['voltage_pooled_rmse_mv','calcium_pooled_rmse_mM','score','passed']].mean())
    print({'sealed_confirmation':False,'synapses_tested':False,'inference_speed_claim':False})


In [ ]:
DOWNLOAD_IN_BROWSER=False
if DOWNLOAD_IN_BROWSER:
    import base64
    from IPython.display import display,Javascript
    encoded=base64.b64encode(Path(ARCHIVE).read_bytes()).decode('ascii')
    display(Javascript('const raw=atob('+json.dumps(encoded)+');const bytes=Uint8Array.from(raw,c=>c.charCodeAt(0));const url=URL.createObjectURL(new Blob([bytes],{type:"application/zip"}));const a=document.createElement("a");a.href=url;a.download="giada_cc_n2_unified_cell.zip";document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),30000);'))
